In [ ]:
import os
import pandas as pd

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode"
]

MODELS = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "google/gemma-3-12b-it",
    "Qwen/Qwen2.5-7B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3"
]

RANDOM_BASE = "../rqs/random_order"
TARGET_BASE = "../rqs/targeted_orders"

OUTPUT_DIR = "order_strategy_comparison_mean"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------------------------
# LOAD RANDOM RESULTS
# -------------------------------------------------

print("\nLoading random permutation results...")

random_rows = []

for repo in REPOS:
    for model in MODELS:

        provider, model_name = model.split("/", 1)

        path = f"{RANDOM_BASE}/{repo}/models/{provider}/{model_name}/logs/order_permutation_results_macro_f1.csv"

        if not os.path.exists(path):
            print("Missing:", path)
            continue

        df = pd.read_csv(path)
        df.columns = [c.strip() for c in df.columns]

        df["Repo"] = repo
        df["Model"] = model

        random_rows.append(df)

if not random_rows:
    raise RuntimeError("No random permutation files found")

random_df = pd.concat(random_rows, ignore_index=True)

random_df["Shuffled_Macro_F1"] = pd.to_numeric(
    random_df["Shuffled_Macro_F1"], errors="coerce"
)

random_df = random_df.dropna(subset=["Shuffled_Macro_F1"])

print("Random rows loaded:", len(random_df))

# -------------------------------------------------
# LOAD TARGETED RESULTS
# -------------------------------------------------

print("\nLoading targeted ordering results...")

target_rows = []

for repo in REPOS:
    for model in MODELS:

        provider, model_name = model.split("/", 1)

        path = f"{TARGET_BASE}/{repo}/models/{provider}/{model_name}/logs/targeted_order_results_macro_f1.csv"

        if not os.path.exists(path):
            print("Missing:", path)
            continue

        df = pd.read_csv(path)
        df.columns = [c.strip() for c in df.columns]

        df["Repo"] = repo
        df["Model"] = model

        target_rows.append(df)

if not target_rows:
    raise RuntimeError("No targeted ordering files found")

target_df = pd.concat(target_rows, ignore_index=True)

target_df["New_Macro_F1"] = pd.to_numeric(
    target_df["New_Macro_F1"], errors="coerce"
)

target_df = target_df.dropna(subset=["New_Macro_F1"])

print("Targeted rows loaded:", len(target_df))

# -------------------------------------------------
# MEAN PERFORMANCE PER CONFIG (ALL k)
# -------------------------------------------------

avg_random = (
    random_df
    .groupby(["Repo", "Model", "FewShot_Count"])["Shuffled_Macro_F1"]
    .mean()
    .reset_index(name="Avg_Random_F1")
)

avg_target = (
    target_df
    .groupby(["Repo", "Model", "FewShot_Count"])["New_Macro_F1"]
    .mean()
    .reset_index(name="Avg_Targeted_F1")
)

# -------------------------------------------------
# MERGE
# -------------------------------------------------

comparison = pd.merge(
    avg_random,
    avg_target,
    on=["Repo", "Model", "FewShot_Count"],
    how="inner"
)

# -------------------------------------------------
# DELTA
# -------------------------------------------------

comparison["Delta_F1"] = (
    comparison["Avg_Targeted_F1"] -
    comparison["Avg_Random_F1"]
)

print("\nTotal configurations (ALL k):", len(comparison))

# -------------------------------------------------
# OVERALL SUMMARY (ALL k)
# -------------------------------------------------

wins_all = (comparison["Delta_F1"] > 0).sum()
losses_all = (comparison["Delta_F1"] < 0).sum()
ties_all = (comparison["Delta_F1"] == 0).sum()

print("\n========================================")
print("ALL k RESULTS")
print("========================================")

print("Targeted better:", wins_all)
print("Random better:", losses_all)
print("Tie:", ties_all)

# -------------------------------------------------
# FILTER TO k = 8
# -------------------------------------------------

comparison_k8 = comparison[
    comparison["FewShot_Count"] == 8
].copy()

print("\nFiltered to k = 8")
print("Configurations (k=8):", len(comparison_k8))

# -------------------------------------------------
# SUMMARY k = 8
# -------------------------------------------------

wins_k8 = (comparison_k8["Delta_F1"] > 0).sum()
losses_k8 = (comparison_k8["Delta_F1"] < 0).sum()
ties_k8 = (comparison_k8["Delta_F1"] == 0).sum()

print("\n========================================")
print("k = 8 RESULTS")
print("========================================")

print("Targeted better:", wins_k8)
print("Random better:", losses_k8)
print("Tie:", ties_k8)

print("\nMean ΔF1:", round(comparison_k8["Delta_F1"].mean(), 4))
print("Median ΔF1:", round(comparison_k8["Delta_F1"].median(), 4))


Loading random permutation results...
Random rows loaded: 5380

Loading targeted ordering results...
Targeted rows loaded: 1280

Total configurations (ALL k): 160

ALL k RESULTS
Targeted better: 33
Random better: 127
Tie: 0

Filtered to k = 8
Configurations (k=8): 20

k = 8 RESULTS
Targeted better: 8
Random better: 12
Tie: 0

Mean ΔF1: -0.0006
Median ΔF1: -0.0032
